# Similarity results — publication figures

Bar charts (mean ± 95% CI across 3 seeds, t-distribution) with paired
Wilcoxon significance vs. ICICLE, one figure per (split × metric). Scaffold
split has baseline (NEIMS/RASSP/MassFormer) comparisons; random split is
ICICLE-only (baselines not yet run on random split — see missing-experiments
note in `PUB_READY_PLAN.md`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from icicle.utils.visualization.eval_plots import (
    load_seed_csvs,
    mean_ci_t,
    plot_metric_bars,
)
from icicle.utils.visualization.style import save_fig, set_style

set_style("manuscript")

RESULTS = Path("/home/magled/icicle-dev/results/eval")
OUTPUT_DIR = Path("figures/similarity")
METRICS = [
    "cosine_similarity",
    "entropy_similarity",
    "weighted_cosine_nist_gc",
    "composite_similarity_nist_gc",
]
METRIC_DISPLAY_NAMES = {
    "cosine_similarity": "Cosine Similarity",
    "entropy_similarity": "Entropy Similarity",
    "weighted_cosine_nist_gc": "Weighted Cosine",
    "composite_similarity_nist_gc": "Composite Similarity",
}

# Reference ceilings (mean similarity) from constitutional-isomer,
# stereoisomer, and replicate comparison analyses, for the 4 metrics plotted
# above. Drawn as vertical lines on every bar chart alongside ICICLE's own
# mean, so baseline bars can be read against how far they sit below both
# ICICLE and the isomer/replicate similarity ceilings.
REF_LINES = {
    "cosine_similarity": {
        "Isomer": 0.2867,
        "Stereoisomer": 0.9118,
        "Replicate": 0.9480,
    },
    "entropy_similarity": {
        "Isomer": 0.4627,
        "Stereoisomer": 0.9177,
        "Replicate": 0.9305,
    },
    "weighted_cosine_nist_gc": {
        "Isomer": 0.4223,
        "Stereoisomer": 0.9121,
        "Replicate": 0.9605,
    },
    "composite_similarity_nist_gc": {
        "Isomer": 0.3990,
        "Stereoisomer": 0.8410,
        "Replicate": 0.9016,
    },
}

## Configuration

`SPLIT_MODELS` maps split → model label → list of per-seed
`results/eval/<run>/similarity_results.csv` paths. Missing seeds (not yet
finished) are dropped automatically by `load_seed_csvs`.

In [ ]:
SPLIT_MODELS = {
    "scaffold": {
        "ICICLE": [
            RESULTS
            / f"final_entropy_scaffold_s{i}_sim"
            / "similarity_results.csv"
            for i in (1, 2, 3)
        ],
        "NEIMS": [
            RESULTS / f"neims_scaffold_s{i}" / "similarity_results.csv"
            for i in (1, 2, 3)
        ],
        "RASSP": [
            RESULTS / f"rassp_scaffold_s{i}" / "similarity_results.csv"
            for i in (1, 2, 3)
        ],
        "MassFormer": [
            RESULTS / f"massformer_scaffold_s{i}" / "similarity_results.csv"
            for i in (1, 2, 3)
        ],
    },
    "scaffold_rassp_subset": {
        # RASSP-native-split subset: ICICLE/NEIMS/MassFormer downsampled from
        # their full-scaffold eval CSVs to RASSP's covered molecules (see
        # examples/scripts/evaluation/filter_to_rassp_subset.py); RASSP itself
        # already ran on this exact split, used unfiltered.
        "ICICLE": [
            RESULTS
            / "rassp_subset_comparison"
            / f"icicle_scaffold_s{i}_sim_rassp_subset.csv"
            for i in (1, 2, 3)
        ],
        "NEIMS": [
            RESULTS
            / "rassp_subset_comparison"
            / f"neims_scaffold_s{i}_sim_rassp_subset.csv"
            for i in (1, 2, 3)
        ],
        "RASSP": [
            RESULTS / f"rassp_scaffold_s{i}" / "similarity_results.csv"
            for i in (1, 2, 3)
        ],
        "MassFormer": [
            RESULTS
            / "rassp_subset_comparison"
            / f"massformer_scaffold_s{i}_sim_rassp_subset.csv"
            for i in (1, 2, 3)
        ],
    },
    "random": {
        "ICICLE": [
            RESULTS
            / f"final_entropy_random_s{i}_sim"
            / "similarity_results.csv"
            for i in (1, 2, 3)
        ],
        "NEIMS": [
            RESULTS / f"neims_random_s{i}" / "similarity_results.csv"
            for i in (1, 2, 3)
        ],
        "RASSP": [
            RESULTS / f"rassp_random_s{i}" / "similarity_results.csv"
            for i in (1, 2, 3)
        ],
        "MassFormer": [
            RESULTS / f"massformer_random_s{i}" / "similarity_results.csv"
            for i in (1, 2, 3)
        ],
    },
    "random_rassp_subset": {
        # Same idea as "scaffold_rassp_subset" but for the random split:
        # ICICLE/NEIMS/MassFormer downsampled to RASSP's random-native split
        # coverage; RASSP itself already ran on this exact split, unfiltered.
        "ICICLE": [
            RESULTS
            / "rassp_subset_comparison"
            / f"icicle_random_s{i}_sim_rassp_subset.csv"
            for i in (1, 2, 3)
        ],
        "NEIMS": [
            RESULTS
            / "rassp_subset_comparison"
            / f"neims_random_s{i}_sim_rassp_subset.csv"
            for i in (1, 2, 3)
        ],
        "RASSP": [
            RESULTS / f"rassp_random_s{i}" / "similarity_results.csv"
            for i in (1, 2, 3)
        ],
        "MassFormer": [
            RESULTS
            / "rassp_subset_comparison"
            / f"massformer_random_s{i}_sim_rassp_subset.csv"
            for i in (1, 2, 3)
        ],
    },
}

## Bar plots — mean ± 95% CI, significance vs. ICICLE

One figure per (split × metric), in four variants:
- all models vs. all-models-minus-RASSP (`no_rassp` suffix)
- with significance brackets vs. without (`nosig` suffix)

A model is dropped from a given split's plots entirely if none of its seed
CSVs exist yet (reported below).


In [ ]:
loaded_by_split = {}
for split, models in SPLIT_MODELS.items():
    model_dfs = {
        label: load_seed_csvs(paths) for label, paths in models.items()
    }
    for label, dfs in model_dfs.items():
        print(f"[{split}] {label}: {len(dfs)}/3 seeds loaded")
    model_dfs = {label: dfs for label, dfs in model_dfs.items() if dfs}
    loaded_by_split[split] = model_dfs

## Generate + save figures

In [ ]:
for split, model_dfs in loaded_by_split.items():
    if "ICICLE" not in model_dfs:
        print(f"[{split}] ICICLE has no results yet, skipping split entirely")
        continue
    for metric in METRICS:
        dfs_with_metric = {
            k: v for k, v in model_dfs.items() if metric in v[0].columns
        }
        if "ICICLE" not in dfs_with_metric:
            continue

        variants = {"": dfs_with_metric}
        if "RASSP" in dfs_with_metric:
            variants["no_rassp"] = {
                k: v for k, v in dfs_with_metric.items() if k != "RASSP"
            }

        for variant_suffix, variant_dfs in variants.items():
            for sig_suffix, show_significance in (
                ("", True),
                ("nosig", False),
            ):
                fig = plot_metric_bars(
                    variant_dfs,
                    metric,
                    reference="ICICLE",
                    ref_lines=REF_LINES.get(metric),
                    show_significance=show_significance,
                )
                fig.axes[0].set_xlabel(
                    METRIC_DISPLAY_NAMES.get(metric, metric)
                )
                name_parts = [f"similarity_{metric}_{split}"]
                if variant_suffix:
                    name_parts.append(variant_suffix)
                if sig_suffix:
                    name_parts.append(sig_suffix)
                save_fig(fig, "_".join(name_parts), OUTPUT_DIR)
                plt.show()
                plt.close(fig)

## Summary table

In [ ]:
rows = []
for split, model_dfs in loaded_by_split.items():
    for label, dfs in model_dfs.items():
        row = {"Split": split, "Model": label, "Seeds": len(dfs)}
        for metric in METRICS:
            seed_means = np.array(
                [df[metric].mean() for df in dfs if metric in df.columns]
            )
            if len(seed_means) == 0:
                row[METRIC_DISPLAY_NAMES.get(metric, metric)] = "—"
                continue
            mean, half_width = mean_ci_t(seed_means)
            row[METRIC_DISPLAY_NAMES.get(metric, metric)] = (
                f"{mean:.3f} ± {half_width:.3f}"
            )
        row["N (total)"] = sum(len(df) for df in dfs)
        rows.append(row)

df_summary = pd.DataFrame(rows)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_summary.to_csv(OUTPUT_DIR / "similarity_summary.csv", index=False)
df_summary

## Export summary to LaTeX

In [ ]:
def export_similarity_latex(
    df, output_path="figures/similarity/similarity_table.tex"
):
    """Export the similarity summary table to a LaTeX table."""
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].str.replace("±", r"$\pm$", regex=False)
    latex = df.to_latex(index=False, escape=False)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        f.write(latex)
    print(f"LaTeX table exported to {output_path}")
    print(latex)


export_similarity_latex(df_summary)

In [ ]:
df_summary_rassp_subset = df_summary[
    df_summary["Split"].isin(["scaffold_rassp_subset", "random_rassp_subset"])
]
df_summary_rassp_subset.to_csv(
    OUTPUT_DIR / "similarity_summary_rassp_subset.csv", index=False
)
export_similarity_latex(
    df_summary_rassp_subset,
    output_path="figures/similarity/similarity_table_rassp_subset.tex",
)
df_summary_rassp_subset

## RASSP-subset table (standalone)

Same rows as the "rassp_subset" split above, pulled into their own table +
LaTeX export for direct use in the RASSP head-to-head section of the paper.

## Best, worst, and average ICICLE-FE ES spectra

Loads predicted and ground-truth spectra from the ICICLE-FE ES evaluation HDF5 and plots mirrored pairs for the 3 best, 3 worst, and 3 median-performing molecules by entropy similarity.

In [ ]:
import h5py
from icicle.utils.visualization.mass_spectra import plot_mirrored_spectra

ICICLE_SIM_DIR = RESULTS / "final_entropy_scaffold_s1_sim"
NIST_META_PATH = Path(
    "/home/magled/icicle-dev/data/NIST2023_GCMS_main/metadata.tsv"
)
NIST_SPLIT_PATH = Path(
    "/home/magled/icicle-dev/data/NIST2023_GCMS_main/splits/scaffold_no_xeno_aas_deduplicated.tsv"
)
METRIC = "entropy_similarity"
N_EXAMPLES = 3

sim_csv = pd.read_csv(ICICLE_SIM_DIR / "similarity_results.csv")
hdf5_path = ICICLE_SIM_DIR / "all_evaluation_spectra.hdf5"

# Join mol_id → full InChIKey via NIST metadata
meta = pd.read_csv(NIST_META_PATH, sep="\t", usecols=["mol_id", "inchi_key"])
splits = pd.read_csv(NIST_SPLIT_PATH, sep="\t")
sim_csv = sim_csv.merge(meta, on="mol_id", how="left")
sim_csv = sim_csv.dropna(subset=["inchi_key", METRIC])

sim_sorted = sim_csv.sort_values(METRIC).reset_index(drop=True)
n = len(sim_sorted)
mid = n // 2

examples = {
    "Best": sim_sorted.tail(N_EXAMPLES).iloc[::-1],
    "Average": sim_sorted.iloc[
        mid - N_EXAMPLES // 2 : mid + N_EXAMPLES // 2 + 1
    ],
    "Worst": sim_sorted.head(N_EXAMPLES),
}

with h5py.File(hdf5_path, "r") as f:
    for group_name, subset in examples.items():
        print(f"\n--- {group_name} (by {METRIC}) ---")
        for _, row in subset.iterrows():
            key = row["inchi_key"]
            if key not in f:
                print(f"  {key} not in HDF5, skipping")
                continue
            true_spec = f[key]["ground_truth_intensities"][:]
            pred_spec = f[key]["predicted_intensities"][:]
            if pred_spec.max() <= 0:
                print(
                    f"  {key} has zero max intensity in prediction, skipping"
                )
                continue
            pred_spec = pred_spec / pred_spec.max()

            score = row[METRIC]
            print(
                f"{group_name} | {METRIC}={score:.3f} | {row['smiles'][:60]}"
            )
            fig = plot_mirrored_spectra(
                true_spec=true_spec,
                pred_spec=pred_spec,
                true_smiles=row["smiles"],
                title="",
            )
            plt.show()
            plt.close(fig)